# 05. Teacher-Student and Self-Distillation Experiment

This notebook trains a student model using both hard labels and soft teacher predictions.
It supports the report sections on Noisy Student and self-distillation.

In [ ]:
# Common setup
from pathlib import Path
import sys
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find project root: works when notebook is run from repo root or from notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    # Walk upward in case the notebook is launched from a subfolder
    for parent in Path.cwd().resolve().parents:
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORT_DIR / "figures"
TABLE_DIR = REPORT_DIR / "tables"
EXP_DIR = PROJECT_ROOT / "experiments" / "notebook_outputs"
for p in [FIG_DIR, TABLE_DIR, EXP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_DIR)

import torch
from torch.utils.data import DataLoader, Dataset

from bioacoustic.dataset import read_metadata, build_class_list, add_stratified_folds, BirdAudioDataset, encode_multihot, make_label_map
from bioacoustic.models import build_model
from bioacoustic.losses import student_distillation_loss, compute_pos_weight
from bioacoustic.training import validate, save_checkpoint, load_checkpoint
from bioacoustic.distillation import generate_teacher_predictions, save_soft_targets, load_soft_targets
from bioacoustic.utils import seed_everything, get_device

seed_everything(42)
DEVICE = get_device(True)
print("Device:", DEVICE)

## Configuration

In [ ]:
CFG = {
    "metadata_path": DATA_DIR / "train_metadata.csv",
    "audio_dir": DATA_DIR / "train_audio",
    "teacher_checkpoint": EXP_DIR / "sed" / "best_sed.pt",
    "output_dir": EXP_DIR / "self_distillation",
    "sample_rate": 32000,
    "duration": 5.0,
    "n_fft": 2048,
    "hop_length": 768,
    "n_mels": 192,
    "f_min": 50,
    "f_max": 15000,
    "backbone": "tf_efficientnet_b0_ns",
    "student_model_type": "sed",
    "batch_size": 12,
    "num_workers": 2,
    "epochs": 3,
    "lr": 1e-4,
    "fold": 0,
    "n_splits": 5,
    "hard_weight": 0.6,
    "focal_gamma": 2.0,
    "label_smoothing": 0.005,
    "max_train_rows": None,
    "max_valid_rows": None,
}
CFG["output_dir"].mkdir(parents=True, exist_ok=True)

## Data and teacher predictions

In [ ]:
df = read_metadata(CFG["metadata_path"])
classes = build_class_list(df)
label_map = make_label_map(classes)
num_classes = len(classes)
if "fold" not in df.columns:
    df = add_stratified_folds(df, n_splits=CFG["n_splits"], seed=42)
train_df = df[df["fold"] != CFG["fold"]].reset_index(drop=True)
valid_df = df[df["fold"] == CFG["fold"]].reset_index(drop=True)
if CFG["max_train_rows"]:
    train_df = train_df.sample(min(CFG["max_train_rows"], len(train_df)), random_state=42).reset_index(drop=True)
if CFG["max_valid_rows"]:
    valid_df = valid_df.sample(min(CFG["max_valid_rows"], len(valid_df)), random_state=42).reset_index(drop=True)

spec_kwargs = dict(n_fft=CFG["n_fft"], hop_length=CFG["hop_length"], n_mels=CFG["n_mels"], f_min=CFG["f_min"], f_max=CFG["f_max"])
train_ds = BirdAudioDataset(train_df, CFG["audio_dir"], classes, sample_rate=CFG["sample_rate"], duration=CFG["duration"], train=True, spectrogram_kwargs=spec_kwargs)
valid_ds = BirdAudioDataset(valid_df, CFG["audio_dir"], classes, sample_rate=CFG["sample_rate"], duration=CFG["duration"], train=False, spectrogram_kwargs=spec_kwargs)
train_loader_no_soft = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)

soft_path = CFG["output_dir"] / "teacher_soft_targets_train.npy"
if soft_path.exists():
    teacher_probs = load_soft_targets(soft_path)
else:
    teacher = build_model("sed", num_classes=num_classes, backbone=CFG["backbone"], in_channels=1, pretrained=False).to(DEVICE)
    if CFG["teacher_checkpoint"].exists():
        load_checkpoint(CFG["teacher_checkpoint"], teacher, map_location=DEVICE)
        teacher_probs = generate_teacher_predictions(teacher, train_loader_no_soft, device=DEVICE)
        save_soft_targets(soft_path, teacher_probs)
    else:
        print("WARNING: teacher checkpoint not found; using hard labels as fallback soft targets.")
        teacher_probs = []
        for _, y in train_loader_no_soft:
            teacher_probs.append(y.numpy())
        teacher_probs = np.concatenate(teacher_probs, axis=0)
        save_soft_targets(soft_path, teacher_probs)

print("Teacher soft targets:", teacher_probs.shape)

## Dataset wrapper with soft targets

In [ ]:
class SoftTargetDataset(Dataset):
    def __init__(self, base_dataset, soft_targets):
        self.base_dataset = base_dataset
        self.soft_targets = torch.tensor(soft_targets, dtype=torch.float32)
        assert len(self.base_dataset) == len(self.soft_targets)
    def __len__(self):
        return len(self.base_dataset)
    def __getitem__(self, idx):
        x, hard = self.base_dataset[idx]
        soft = self.soft_targets[idx]
        return x, hard, soft

soft_train_ds = SoftTargetDataset(train_ds, teacher_probs)
train_loader = DataLoader(soft_train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"], pin_memory=True)

## Train student with hard-label and soft-label loss

In [ ]:
student = build_model(CFG["student_model_type"], num_classes=num_classes, backbone=CFG["backbone"], in_channels=1, pretrained=True).to(DEVICE)

target_matrix = []
for _, row in train_df.iterrows():
    target_matrix.append(encode_multihot(row["primary_label"], row.get("secondary_labels", None), label_map, include_secondary=True))
target_matrix = torch.tensor(np.stack(target_matrix), dtype=torch.float32)
pos_weight = compute_pos_weight(target_matrix, max_weight=20.0).to(DEVICE)
optimizer = torch.optim.AdamW(student.parameters(), lr=CFG["lr"], weight_decay=1e-4)

logs = []
best_auc = -1.0
for epoch in range(1, CFG["epochs"] + 1):
    student.train()
    losses = []
    for x, hard, soft in train_loader:
        x = x.to(DEVICE).float()
        hard = hard.to(DEVICE).float()
        soft = soft.to(DEVICE).float()
        optimizer.zero_grad(set_to_none=True)
        out = student(x)
        loss = student_distillation_loss(
            out["clip_logits"],
            hard,
            soft,
            hard_weight=CFG["hard_weight"],
            pos_weight=pos_weight,
            gamma=CFG["focal_gamma"],
            label_smoothing=CFG["label_smoothing"],
        )
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))

    val_out = validate(student, valid_loader, loss_fn=None, device=DEVICE)
    metrics = val_out["metrics"]
    row = {"epoch": epoch, "train_loss": float(np.mean(losses)), **metrics}
    logs.append(row)
    print(row)
    score = metrics.get("macro_auc", float("nan"))
    if score == score and score > best_auc:
        best_auc = score
        save_checkpoint(CFG["output_dir"] / "best_student.pt", student, optimizer, epoch=epoch, metrics=metrics)

log_df = pd.DataFrame(logs)
display(log_df)
log_df.to_csv(CFG["output_dir"] / "training_log.csv", index=False)
np.save(CFG["output_dir"] / "valid_y_true.npy", val_out["y_true"])
np.save(CFG["output_dir"] / "valid_y_pred.npy", val_out["y_pred"])

## Plot student metrics

In [ ]:
if len(log_df):
    plt.figure(figsize=(8, 4))
    plt.plot(log_df["epoch"], log_df["train_loss"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Student loss")
    plt.title("Self-distillation student training loss")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "student_distillation_loss.png", dpi=200)
    plt.show()

    if "macro_auc" in log_df:
        plt.figure(figsize=(8, 4))
        plt.plot(log_df["epoch"], log_df["macro_auc"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Macro AUC")
        plt.title("Student validation macro AUC")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "student_auc_curve.png", dpi=200)
        plt.show()